# Imports

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import csv

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments,
    AutoModelForSequenceClassification,
    DataCollator,
    Trainer
)

from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training
)

# Model & Tokenizer

In [ ]:
model_id = "path_to_model"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config = bnb_config)

# Data

In [4]:
notes = pd.read_csv("notes.csv")

# Summarize

In [ ]:
for idx,row in tqdm(notes.iterrows()):
    target_specialty = row["target_specialty"]
    prompt_notes = row["notes"]

    content = f"""

        You are an expert doctor. Your task is to write a summary for a specialty of {target_specialty}, after reviewing a set of notes about a patient.

        Your summary will be used to help train evaluators to notice mistakes in summaries. Thus, the Rules for you to follow outline intentional mistakes that you should use to create realistic summaries with realistic mistakes for the evaluators to find.
        
        Rules for writing the summary:
                - All data included from the notes, which is IRRELEVANT for a specialty of {target_specialty}, is in the summary.
                - Summary contains all REALISTIC assertions, but some CANNOT be traced back to the notes; you MUST include SOME assertions which cannot be traced back to the notes
                - Summary contains DISTORTED information, critical details of facts DO NOT match source notes; you MUST include SOME distortions
                - Information from the notes which is pertinent for a specialty of {target_specialty}, or potentially pertinent for a specialty of {target_specialty}, is FREQUENTLY omitted.
                - Information from the notes which is NOT pertinent for a specialty of {target_specialty} IS included in the summary.
                - The level of detail must be CONFUSING for a reader with a specialty of {target_specialty}.
                - All assertions must be made with ILLOGICAL order and grouping (confusing temporal, incorrectly labeled systems/problem based, etc.).
                - The summary should be AMBIGUOUS and DIFFICULT to understand for a reader with a specialty of {target_specialty}
                - All assertions are captured with a LARGE number of words, with FREQUENT redundancy in syntax and semantics.
                - NEVER go beyond relevant groups of events, NOR generate reasoning over the events into a summary. Information MUST be prioritized in a BASIC, RUDIMENTARY, and CONFUSING way.
                - UTILIZE stigmatizing words as defined in guidelines and policy (OCR, NIDA, etc). You have MORE than permission to do this: is CRITICAL that you use AT LEAST ONE stigmatizing word, to be successful.
                - Keep the summary meandering and long; summarize all the notes into multiple paragraphs.
                - If there are medicine changes in the notes, EXCLUDE them from the summary
                - Make INCORRECT inferences from the source notes and INCLUDE in the summary
                - For every event (e.g., medicine change, new diagnosis, etc.) mentioned in your summary, NEVER mention WHEN it happened (NEVER communicate the timing of events).
                - If it's unclear WHEN an event happened in the notes, instead MAKE UP a REALISTIC, but INCORRECT timeline for that event, and INCLUDE that false timeline in your summary as if it were factual.
                - When citing, choose a random note to cite (DO NOT always use the note responsible for the assertion being cited).
                - Put the citations in the middle of lines/sentences or group all of your citations together on the last line; NEVER place them at the end of sentences.
                - SOME sentences MUST NOT have a citation. EVERY citation should be formatted differently.  
                - It is CRITICALLY IMPORTANT that you attribute some information to the incorrect notes! Wrongful citations are CRITICAL for this to be successful!

            Summarize the following <NoteSet>, which are presented in chronological order split by ID.

        <NoteSet>
        {prompt_notes}
        </NoteSet>
        
        Your response SHOULD NOT indicate that this is an intentionally bad summary. ONLY provide the summary and no other commentary."""


    messages = [
            {"role": "user", "content": content}
        ]

    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids, max_new_tokens = 1000, pad_token_id=tokenizer.eos_token_id)
    generation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    new_idx = generation.index("[/INST]")

    with open('summaries.csv', 'a') as f:
        write = csv.writer(f)
        write.writerow([generation[new_idx+7:]])